# Flight delay prediction (ANAC VRA + NOAA GSOD) on Google Colab

This notebook only **installs the package and runs the pipeline**: all logic lives in
`src/flight_delay`. Data layers and reports persist in Google Drive
(`/content/drive/MyDrive/Dados_ANAC`), so a disconnected session resumes where it stopped.

First run? Keep `TEST_RUN = True` (2 months, small sample) to validate the environment, then set it to `False`.

In [ ]:
BRANCH = "refactor/architecture"   # or "main" after the merge
TEST_RUN = True                     # True = configs/test.yaml on top of base.yaml

REPO_DIR = "/content/flight-delay-prediction-brazil"
!test -d {REPO_DIR} || git clone -q -b {BRANCH} https://github.com/pedrohenriquesantanadarocha-stack/flight-delay-prediction-brazil.git {REPO_DIR}
!git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q
%pip install -q "{REPO_DIR}[all]"
%pip install -q --force-reinstall --no-deps "{REPO_DIR}"   # always pick up the latest code

In [ ]:
from flight_delay.adapters import colab
colab.setup()   # mounts Google Drive and authenticates BigQuery (weather)

In [ ]:
import logging, warnings
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", datefmt="%H:%M:%S", force=True)
warnings.filterwarnings("ignore", category=FutureWarning)

from flight_delay.config import load_settings
from flight_delay.pipelines.run import run

configs = [f"{REPO_DIR}/configs/base.yaml", f"{REPO_DIR}/configs/colab.yaml"]
if TEST_RUN:
    configs.append(f"{REPO_DIR}/configs/test.yaml")
settings = load_settings(configs)
print(settings.scope)

In [ ]:
results = run(settings, show_plots=True)

`results` holds the trained models (`results["results"]`), the tables (`results["test_table"]`,
`results["ablation"]`, ...) and the report folder (`results["report_dir"]`) with the figures, CSVs and
`tabelas_modelagem.tex`.